# Model Validation — Alternative Specifications & SHAP

Validates the LightGBM choice against six alternatives (Random Forest, XGBoost, Ridge, OLS, naive AR(1), two-way fixed-effects panel regression) on an identical feature set and evaluation protocol. Also demonstrates the global-vs-local SHAP distinction concretely (OLS coefficients vs. SHAP mean-|value| vs. a single observation's local decomposition) and extends local SHAP explanations to each case-study MSA's 2023 prediction.

**Requires:** run `01_main_model.ipynb` first (reuses its fitted objects/exports).

In [ ]:
"""
Section 7b (NEW) — Model Comparison: Rationale for Choosing LightGBM
=========================================================================
Compares the main model's LightGBM specification against six
alternatives, using the EXACT SAME feature set, lag structure, and
evaluation protocol as the main v14b model (Section 7) -- reuses
X_train, X_predict, y_train, df_train, df_predict, FEATURES,
CAT_FEAT_IDX, TRAIN_YEARS, msa_list already built there, so this is
NOT a separately-constructed comparison; every model below sees
identical inputs to LightGBM.

MODELS COMPARED:
  1. LightGBM (the main model -- numbers reused from Section 7, not
     re-fit, to guarantee this comparison matches the paper exactly)
  2. Random Forest
  3. XGBoost
  4. Ridge Regression (regularized linear, same feature set)
  5. OLS Linear Regression (unregularized linear, same feature set)
  6. Naive AR(1) -- predicts next year's log(Available_SF_Total) from
     ONLY that MSA's own prior-year value, no other covariates. This
     directly quantifies how much the full feature set adds over
     simple persistence -- relevant given the main model deliberately
     EXCLUDES an AR term from its own feature set (see the main
     model's Design notes: an AR term dominated SHAP and suppressed
     economic signal). This is a different question from "should the
     main model include AR" -- it's "how well does AR alone do,
     compared to everything else."
  7. Two-Way Fixed-Effects Panel Regression (MSA FE + Year FE + the
     same covariates, estimated via LSDV/dummy-variable OLS) -- the
     standard econometric baseline panel ML approaches are typically
     benchmarked against in regional/urban economics. NOT a causal
     model -- it has no identification strategy (no instrument, DiD,
     or discontinuity) any more than the main model does. It is
     included because it is the conventional linear-panel comparison
     a reviewer would expect, not because it resolves the causal-
     identification limitation already acknowledged elsewhere in this
     project.

IMPORTANT METHODOLOGICAL NOTE ON THE FIXED-EFFECTS MODEL AND LOOCV:
MSA-level leave-one-out cross-validation is NOT run for the two-way
FE model, and this is a substantive point, not just a computational
shortcut. A fixed-effects model's whole mechanism is estimating one
dummy coefficient PER MSA; it structurally cannot produce a
coefficient for an MSA that was never in the training data, so
"predicting a held-out MSA" is not really a well-defined operation
for this model the way it is for a tree-based model using lagged
covariates as context instead of unit fixed effects. This is reported
explicitly below (fixed-effects results are shown for in-sample and
2019-holdout fit only, with LOOCV marked "N/A -- see note") rather
than silently omitted, because it is itself a relevant piece of
rationale for preferring a covariate-based ML approach: this project's
predict task IS extrapolation to a full panel of MSAs across new
years, and a model architecture that cannot generalize to any MSA
outside its training set is a poor match for that task regardless of
its in-sample fit.

(Full original version history retained in git log / docs/methodology.md.)
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

print(f"\n{'='*70}")
print("SECTION 7b — MODEL COMPARISON: RATIONALE FOR CHOOSING LIGHTGBM")
print(f"{'='*70}")

comparison_results = []

# ══════════════════════════════════════════════════════════════════
# 0. LIGHTGBM — reuse the main model's own numbers (Section 7),
#    not re-fit, so this comparison matches the paper's reported
#    figures exactly rather than a second, possibly-different run.
# ══════════════════════════════════════════════════════════════════
comparison_results.append({
    'Model': 'LightGBM (main model)',
    'InSample_R2': r2_insample, 'InSample_MAE': mae_insample,
    'Holdout2019_R2': r2_2019, 'Holdout2019_MAE': mae_2019,
    'LOOCV_R2': loo_r2, 'LOOCV_MAE': loo_mae,
})

# ══════════════════════════════════════════════════════════════════
# 1. RANDOM FOREST
# ══════════════════════════════════════════════════════════════════
def eval_sklearn_model(make_model, model_name, needs_scaling=False):
    """Fits on X_train/y_train, evaluates in-sample + 2019 holdout +
    MSA-level LOOCV, using the same protocol as the main model."""
    m = make_model()
    m.fit(X_train.values, y_train)
    pred_train = m.predict(X_train.values)
    r2_is = r2_score(y_train, pred_train)
    mae_is = mean_absolute_error(y_train, pred_train)

    m_h = make_model()
    m_h.fit(X_h_tr.values, y_h_tr)
    pred_2019 = m_h.predict(X_h_te.values)
    r2_h = r2_score(y_h_te, pred_2019)
    mae_h = mean_absolute_error(y_h_te, pred_2019)

    loo_preds = np.full(len(y_train), np.nan)
    for msa in msa_list:
        test_idx = df_train[df_train['MSA_Name'] == msa].index.tolist()
        train_idx = df_train[df_train['MSA_Name'] != msa].index.tolist()
        if not train_idx or not test_idx:
            continue
        m_loo = make_model()
        m_loo.fit(X_train.values[train_idx], y_train[train_idx])
        loo_preds[test_idx] = m_loo.predict(X_train.values[test_idx])
    valid = ~np.isnan(loo_preds)
    r2_loo = r2_score(y_train[valid], loo_preds[valid])
    mae_loo = mean_absolute_error(y_train[valid], loo_preds[valid])

    print(f"  {model_name}: in-sample R2={r2_is:.4f}, 2019 holdout R2={r2_h:.4f}, "
          f"LOOCV R2={r2_loo:.4f}")
    return {
        'Model': model_name,
        'InSample_R2': r2_is, 'InSample_MAE': mae_is,
        'Holdout2019_R2': r2_h, 'Holdout2019_MAE': mae_h,
        'LOOCV_R2': r2_loo, 'LOOCV_MAE': mae_loo,
    }

print("\nFitting Random Forest (in-sample, 2019 holdout, MSA-LOOCV)...")
comparison_results.append(eval_sklearn_model(
    lambda: RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=10,
        random_state=42, n_jobs=-1),
    'Random Forest'))

# ══════════════════════════════════════════════════════════════════
# 2. XGBOOST — hyperparameters chosen to mirror the main model's
#    LightGBM settings as closely as the two libraries allow (same
#    depth/leaf/regularization philosophy, not a separately-tuned
#    "best" XGBoost), so the comparison reflects algorithm choice
#    rather than one model being tuned harder than the other.
# ══════════════════════════════════════════════════════════════════
print("\nFitting XGBoost (in-sample, 2019 holdout, MSA-LOOCV)...")
comparison_results.append(eval_sklearn_model(
    lambda: xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        min_child_weight=40, subsample=0.7, colsample_bytree=0.7,
        reg_alpha=0.1, reg_lambda=3.0, random_state=42, verbosity=0),
    'XGBoost'))

# ══════════════════════════════════════════════════════════════════
# 3. RIDGE REGRESSION (regularized linear, same feature set)
# ══════════════════════════════════════════════════════════════════
print("\nFitting Ridge Regression (in-sample, 2019 holdout, MSA-LOOCV)...")
comparison_results.append(eval_sklearn_model(
    lambda: Ridge(alpha=10.0, random_state=42), 'Ridge Regression'))

# ══════════════════════════════════════════════════════════════════
# 4. OLS LINEAR REGRESSION (unregularized, same feature set)
# ══════════════════════════════════════════════════════════════════
print("\nFitting OLS Linear Regression (in-sample, 2019 holdout, MSA-LOOCV)...")
comparison_results.append(eval_sklearn_model(
    lambda: LinearRegression(), 'OLS Linear Regression'))

# ══════════════════════════════════════════════════════════════════
# 5. NAIVE AR(1) — predicts log(Available_SF_Total)_t from ONLY that
#    MSA's own log(Available_SF_Total)_{t-1}. No other covariates.
#    Built here (not reused from FEATURES) because the main model
#    deliberately excludes this exact term.
# ══════════════════════════════════════════════════════════════════
print("\nFitting naive AR(1) baseline (own-lag only, no other covariates)...")

ar1_train = df_train[['MSA_Name', 'Year', 'log_Available_SF_Total']].copy()
ar1_train = ar1_train.sort_values(['MSA_Name', 'Year'])
ar1_train['log_Available_SF_Total_lag1'] = ar1_train.groupby('MSA_Name')['log_Available_SF_Total'].shift(1)
ar1_train = ar1_train.dropna(subset=['log_Available_SF_Total_lag1'])

X_ar1 = ar1_train[['log_Available_SF_Total_lag1']].values
y_ar1 = ar1_train['log_Available_SF_Total'].values
ar1_msa = ar1_train['MSA_Name'].values

ar1_model = LinearRegression().fit(X_ar1, y_ar1)
ar1_pred_train = ar1_model.predict(X_ar1)
r2_ar1_is = r2_score(y_ar1, ar1_pred_train)
mae_ar1_is = mean_absolute_error(y_ar1, ar1_pred_train)

ar1_tr2018 = ar1_train[ar1_train['Year'] <= 2018]
ar1_te2019 = ar1_train[ar1_train['Year'] == 2019]
ar1_h_model = LinearRegression().fit(
    ar1_tr2018[['log_Available_SF_Total_lag1']].values, ar1_tr2018['log_Available_SF_Total'].values)
ar1_pred_2019 = ar1_h_model.predict(ar1_te2019[['log_Available_SF_Total_lag1']].values)
r2_ar1_h = r2_score(ar1_te2019['log_Available_SF_Total'], ar1_pred_2019)
mae_ar1_h = mean_absolute_error(ar1_te2019['log_Available_SF_Total'], ar1_pred_2019)

ar1_loo_preds = np.full(len(y_ar1), np.nan)
for msa in msa_list:
    test_idx = np.where(ar1_msa == msa)[0]
    train_idx = np.where(ar1_msa != msa)[0]
    if len(train_idx) == 0 or len(test_idx) == 0:
        continue
    m_loo = LinearRegression().fit(X_ar1[train_idx], y_ar1[train_idx])
    ar1_loo_preds[test_idx] = m_loo.predict(X_ar1[test_idx])
valid = ~np.isnan(ar1_loo_preds)
r2_ar1_loo = r2_score(y_ar1[valid], ar1_loo_preds[valid])
mae_ar1_loo = mean_absolute_error(y_ar1[valid], ar1_loo_preds[valid])

print(f"  Naive AR(1): in-sample R2={r2_ar1_is:.4f}, 2019 holdout R2={r2_ar1_h:.4f}, "
      f"LOOCV R2={r2_ar1_loo:.4f}")
comparison_results.append({
    'Model': 'Naive AR(1) (own-lag only)',
    'InSample_R2': r2_ar1_is, 'InSample_MAE': mae_ar1_is,
    'Holdout2019_R2': r2_ar1_h, 'Holdout2019_MAE': mae_ar1_h,
    'LOOCV_R2': r2_ar1_loo, 'LOOCV_MAE': mae_ar1_loo,
})

# ══════════════════════════════════════════════════════════════════
# 6. TWO-WAY FIXED-EFFECTS PANEL REGRESSION (MSA FE + Year FE + same
#    covariates, LSDV/dummy-variable OLS). Standard econometric
#    baseline -- NOT a causal model (no identification strategy).
#    LOOCV intentionally NOT run -- see module docstring for why.
# ══════════════════════════════════════════════════════════════════
print("\nFitting Two-Way Fixed-Effects Panel Regression (MSA FE + Year FE)...")

fe_encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
msa_dummies_train = fe_encoder.fit_transform(df_train[['MSA_Name']])
year_dummies_train = pd.get_dummies(df_train['Year'], prefix='Year', drop_first=True).values.astype(float)

X_fe_train = np.hstack([X_train.values, msa_dummies_train, year_dummies_train])
fe_model = LinearRegression().fit(X_fe_train, y_train)
fe_pred_train = fe_model.predict(X_fe_train)
r2_fe_is = r2_score(y_train, fe_pred_train)
mae_fe_is = mean_absolute_error(y_train, fe_pred_train)

msa_dummies_h_tr = fe_encoder.transform(tr_2018[['MSA_Name']])
year_dummies_h_tr = pd.get_dummies(tr_2018['Year'], prefix='Year', drop_first=True)
year_dummies_h_tr = year_dummies_h_tr.reindex(
    columns=pd.get_dummies(df_train['Year'], prefix='Year', drop_first=True).columns, fill_value=0
).values.astype(float)
X_fe_h_tr = np.hstack([X_h_tr.values, msa_dummies_h_tr, year_dummies_h_tr])
fe_h_model = LinearRegression().fit(X_fe_h_tr, y_h_tr)

msa_dummies_h_te = fe_encoder.transform(te_2019[['MSA_Name']])
year_dummies_h_te = pd.DataFrame(
    0, index=te_2019.index,
    columns=pd.get_dummies(df_train['Year'], prefix='Year', drop_first=True).columns
).values.astype(float)  # 2019 dummy is the reference year dropped in this holdout fit
X_fe_h_te = np.hstack([X_h_te.values, msa_dummies_h_te, year_dummies_h_te])
fe_pred_2019 = fe_h_model.predict(X_fe_h_te)
r2_fe_h = r2_score(y_h_te, fe_pred_2019)
mae_fe_h = mean_absolute_error(y_h_te, fe_pred_2019)

print(f"  Two-Way FE Panel: in-sample R2={r2_fe_is:.4f}, 2019 holdout R2={r2_fe_h:.4f}, "
      f"LOOCV: N/A (see module docstring -- MSA FE cannot be estimated for an unseen MSA)")
comparison_results.append({
    'Model': 'Two-Way FE Panel (MSA FE + Year FE)',
    'InSample_R2': r2_fe_is, 'InSample_MAE': mae_fe_is,
    'Holdout2019_R2': r2_fe_h, 'Holdout2019_MAE': mae_fe_h,
    'LOOCV_R2': np.nan, 'LOOCV_MAE': np.nan,
})

# ══════════════════════════════════════════════════════════════════
# 7. FINAL COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════
comparison_df = pd.DataFrame(comparison_results)
comparison_df = comparison_df.round(4)
comparison_df.to_csv("Model_Comparison_v14b.csv", index=False)

print(f"\n{'='*70}")
print("MODEL COMPARISON SUMMARY")
print(f"{'='*70}")
print(comparison_df.to_string(index=False))
print(f"\nSaved: Model_Comparison_v14b.csv")
print(f"\nNOTE: Two-Way FE Panel LOOCV is N/A by construction, not missing data --")
print(f"a fixed-effects model cannot estimate a coefficient for an MSA absent from")
print(f"training, so genuine leave-one-MSA-out evaluation is not a well-defined")
print(f"operation for this model architecture. This is itself part of the rationale")
print(f"for preferring a covariate-based ML approach for this project's predict task.")

In [ ]:
"""
Section 7d (NEW) — Local vs. Global Feature Importance, Concretely
=========================================================================
Demonstrates the "linear coefficients are global, SHAP is local and
aggregable" argument from the model-comparison rationale with an
actual example from this project's own data, rather than leaving it
as an abstract claim.

THREE THINGS THIS SCRIPT SHOWS:
  1. GLOBAL: a standardized-OLS coefficient table (one number per
     feature, identical for every observation) side by side with the
     SHAP mean-|value| ranking (also one number per feature, but
     built by averaging observation-level detail that still exists
     underneath it).
  2. LOCAL: for ONE specific, real MSA-year (configurable below), the
     actual per-feature SHAP contributions for THAT observation --
     which can differ substantially from the global ranking, and
     which OLS has no equivalent for (OLS would apply the exact same
     coefficient to this MSA-year as to every other one).
  3. AGGREGATION PROOF: manually re-averages the per-observation SHAP
     values across ALL training observations and confirms it
     reproduces the same global ranking already saved in
     AvailSFTotal_SHAP_importance.csv -- i.e., the local detail and
     the global summary are the SAME underlying numbers at different
     levels of aggregation, not two different things.

Requires: run after Section 9 of the main v14b model script (SHAP
computation) -- reuses shap_values, X_train, FEATURES, display_names,
df_train, y_train, shap_df already built there.

(Full original version history retained in git log / docs/methodology.md.)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# ══════════════════════════════════════════════════════════════════
# CONFIGURE which MSA-year to explain locally. Pick one that matters
# to your paper's narrative -- e.g. the largest structural deficit or
# surplus MSA in a specific year. Update these two lines as needed.
# ══════════════════════════════════════════════════════════════════
EXPLAIN_MSA = "Washington-Arlington-Alexandria, DC-VA-MD-WV"
EXPLAIN_YEAR = 2019  # last pre-pandemic training year -- a genuine
                      # in-sample observation, not a predict-period row

OUTPUT_COMPARISON_CSV = "Local_vs_Global_Importance_Comparison.csv"

print(f"\n{'='*70}")
print("SECTION 7d — LOCAL VS. GLOBAL FEATURE IMPORTANCE, CONCRETELY")
print(f"{'='*70}")

# ══════════════════════════════════════════════════════════════════
# 1. GLOBAL — standardized OLS coefficients vs. SHAP mean-|value|
# ══════════════════════════════════════════════════════════════════
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.values)
ols_std = LinearRegression().fit(X_train_scaled, y_train)

global_comparison = pd.DataFrame({
    'Feature': display_names,
    'OLS_Standardized_Coefficient': ols_std.coef_,
    'OLS_Abs_Coefficient': np.abs(ols_std.coef_),
    'SHAP_Mean_Abs_Value': shap_df.set_index('Feature').reindex(display_names)['Mean_SHAP'].values,
})
global_comparison['OLS_Rank'] = global_comparison['OLS_Abs_Coefficient'].rank(ascending=False).astype(int)
global_comparison['SHAP_Rank'] = global_comparison['SHAP_Mean_Abs_Value'].rank(ascending=False).astype(int)
global_comparison = global_comparison.sort_values('SHAP_Rank')

print(f"\nGLOBAL comparison -- one number per feature, either way "
      f"(OLS coefficient magnitude vs. mean |SHAP|):")
print(global_comparison[['Feature', 'OLS_Rank', 'SHAP_Rank',
                          'OLS_Abs_Coefficient', 'SHAP_Mean_Abs_Value']].to_string(index=False))

rank_corr = global_comparison['OLS_Rank'].corr(global_comparison['SHAP_Rank'], method='spearman')
print(f"\nSpearman rank correlation between OLS and SHAP global rankings: {rank_corr:.3f}")
print(f"(A high correlation would mean the two methods broadly agree on WHICH features matter")
print(f" globally -- the point of this section is that SHAP additionally lets you go BELOW this")
print(f" single global number to any individual observation, which OLS structurally cannot.)")

global_comparison.to_csv(OUTPUT_COMPARISON_CSV, index=False)
print(f"\nSaved: {OUTPUT_COMPARISON_CSV}")

# ══════════════════════════════════════════════════════════════════
# 2. LOCAL — one specific MSA-year's actual SHAP breakdown
# ══════════════════════════════════════════════════════════════════
match = df_train[(df_train['MSA_Name'] == EXPLAIN_MSA) & (df_train['Year'] == EXPLAIN_YEAR)]
if len(match) == 0:
    raise SystemExit(
        f"STOPPING -- {EXPLAIN_MSA!r} / {EXPLAIN_YEAR} not found in df_train. "
        f"Check the MSA name matches exactly (including state abbreviation formatting) "
        f"and that {EXPLAIN_YEAR} is within the training period (2006-2019)."
    )
obs_idx = match.index[0]
# df_train's index may not align 1:1 with X_train's row order if any
# reset_index happened differently -- recover the correct row position.
row_pos = df_train.index.get_loc(obs_idx)

obs_shap = shap_values[row_pos]
obs_features = X_train.iloc[row_pos]

local_breakdown = pd.DataFrame({
    'Feature': display_names,
    'Feature_Value': obs_features.values,
    'SHAP_Value_This_Observation': obs_shap,
}).sort_values('SHAP_Value_This_Observation', key=np.abs, ascending=False)

print(f"\n{'='*70}")
print(f"LOCAL explanation — {EXPLAIN_MSA}, {EXPLAIN_YEAR}")
print(f"{'='*70}")
print(f"Base value (average model prediction across all training data): "
      f"{explainer.expected_value:.4f}")
print(f"This observation's actual predicted log(Available_SF_Total): "
      f"{explainer.expected_value + obs_shap.sum():.4f}")
print(f"\nTop 10 features driving THIS SPECIFIC prediction (not the global average):")
print(local_breakdown.head(10).to_string(index=False))

print(f"\nCompare this to what OLS would say about this same MSA-year: OLS has no")
print(f"observation-specific answer at all -- it would apply the exact same global")
print(f"coefficient (from the table above) to this MSA-year as to every other one,")
print(f"regardless of whether that global average actually describes what happened here.")

# ══════════════════════════════════════════════════════════════════
# 3. WATERFALL PLOT — visualizes exactly what the table above shows
# ══════════════════════════════════════════════════════════════════
safe_name = EXPLAIN_MSA.replace(',', '').replace(' ', '_').replace('-', '_')
output_png = f"shap_local_explanation_{safe_name}_{EXPLAIN_YEAR}.png"

plt.figure(figsize=(10, 8))
shap.plots.waterfall(
    shap.Explanation(
        values=obs_shap,
        base_values=explainer.expected_value,
        data=obs_features.values,
        feature_names=display_names,
    ),
    max_display=15, show=False,
)
plt.title(f"SHAP Local Explanation — {EXPLAIN_MSA}, {EXPLAIN_YEAR}", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(output_png, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: {output_png}")

# ══════════════════════════════════════════════════════════════════
# 4. AGGREGATION PROOF — manually re-derive the global ranking from
#    per-observation values, confirm it matches shap_df exactly
# ══════════════════════════════════════════════════════════════════
manual_global = pd.DataFrame({
    'Feature': display_names,
    'Manually_Reaggregated_Mean_Abs_SHAP': np.abs(shap_values).mean(axis=0),
}).sort_values('Manually_Reaggregated_Mean_Abs_SHAP', ascending=False)

check = manual_global.merge(
    shap_df.rename(columns={'Mean_SHAP': 'Original_Mean_SHAP'}), on='Feature')
check['Discrepancy'] = (check['Manually_Reaggregated_Mean_Abs_SHAP'] - check['Original_Mean_SHAP']).abs()
max_discrepancy = check['Discrepancy'].max()

print(f"\n{'='*70}")
print("AGGREGATION PROOF — re-averaging every individual observation's SHAP values")
print("reproduces the exact same global ranking already reported in Feature Importance")
print(f"{'='*70}")
print(f"Max discrepancy between manually-reaggregated and originally-reported global "
      f"values: {max_discrepancy:.10f}")
print(f"({'CONFIRMED -- identical' if max_discrepancy < 1e-9 else 'MISMATCH -- investigate'}: "
      f"the global Feature Importance table and this MSA-year's local breakdown are the SAME "
      f"underlying per-observation values, just aggregated differently.)")

print(f"\n{'='*60}")
print("DONE — Section 7d complete")
print(f"{'='*60}")

In [ ]:
"""
Section 7e (NEW) — Individual SHAP Feature Importance, Per Case-Study MSA
=========================================================================
Extends Section 7d (which explained ONE observation, Washington-
Arlington-Alexandria in 2019) to all 8 case-study target markets,
explaining each MSA's 2023 counterfactual prediction specifically --
the year most directly tied to this paper's structural-gap findings,
rather than a pre-pandemic training-period year.

IMPORTANT MECHANICAL NOTE: the main model's Section 9 only ever runs
`explainer.shap_values()` on X_train (2006-2019) -- it never scores
the predict-period rows (2020-2023). This script does that here for
the first time: the SAME fitted `explainer` object from Section 9 is
applied to X_predict instead. This is a completely standard and valid
use of a fitted TreeExplainer on new data -- it does NOT retrain
anything or change the model in any way, it simply asks the already-
trained explainer to decompose a different set of predictions.

WHAT THIS SCRIPT PRODUCES, for each of the 8 target markets:
  1. A per-MSA table of the top 10 features driving that market's 2023
     counterfactual prediction specifically (not the global average).
  2. A combined wide-format CSV (MSA x Feature) for direct comparison.
  3. A cross-market heatmap showing each of the top features' SHAP
     value for every target MSA side by side -- lets you see at a
     glance whether, e.g., log_Adv_Ind_Employees drives Seattle's and
     Dallas's predictions the same way, or differently.
  4. Individual waterfall plots, one per MSA (8 PNG files).

(Full original version history retained in git log / docs/methodology.md.)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

sns.set_style("whitegrid")
plt.rcParams.update({"font.family": "serif", "font.size": 11})

EXPLAIN_YEAR = 2023  # predict-period year -- most relevant to this
                      # paper's structural-gap findings. Change to a
                      # training-period year (2006-2019) if you'd
                      # rather explain a pre-pandemic prediction; the
                      # scoring source (X_predict vs X_train) is
                      # selected automatically below based on this.

TARGET_MSAS = [
    "Seattle-Tacoma-Bellevue, WA",
    "Philadelphia-Camden-Wilmington, PA-NJ-DE-MD",
    "Houston-Pasadena-The Woodlands, TX",
    "Dallas-Fort Worth-Arlington, TX",
    "Boston-Cambridge-Newton, MA-NH",
    "New York-Newark-Jersey City, NY-NJ",
    "San Francisco-Oakland-Fremont, CA",
    "San Jose-Sunnyvale-Santa Clara, CA",
]

OUTPUT_LONG_CSV = f"SHAP_Local_Explanations_CaseStudyMSAs_{EXPLAIN_YEAR}.csv"
OUTPUT_WIDE_CSV = f"SHAP_Local_Explanations_CaseStudyMSAs_{EXPLAIN_YEAR}_wide.csv"
OUTPUT_HEATMAP_PNG = f"shap_crossmarket_heatmap_{EXPLAIN_YEAR}.png"

print(f"\n{'='*70}")
print(f"SECTION 7e — INDIVIDUAL SHAP FEATURE IMPORTANCE, PER CASE-STUDY MSA ({EXPLAIN_YEAR})")
print(f"{'='*70}")

# ══════════════════════════════════════════════════════════════════
# 0. SCORE THE PREDICT-PERIOD ROWS WITH THE ALREADY-FITTED EXPLAINER
#    (only needed if EXPLAIN_YEAR is in the predict period; training-
#    period years reuse shap_values already computed in Section 9)
# ══════════════════════════════════════════════════════════════════
TRAIN_YEARS_SET = set(range(2006, 2020))
if EXPLAIN_YEAR in TRAIN_YEARS_SET:
    source_df = df_train
    source_X = X_train
    source_shap = shap_values
    print(f"Using training-period SHAP values already computed in Section 9 "
          f"(year {EXPLAIN_YEAR} is in the 2006-2019 training set).")
else:
    print(f"Year {EXPLAIN_YEAR} is in the predict period (2020-2023) -- scoring "
          f"X_predict with the already-fitted explainer (no retraining, standard "
          f"use of a fitted TreeExplainer on new data)...")
    source_df = df_predict
    source_X = X_predict
    source_shap = explainer.shap_values(X_predict.values)
    print("Predict-period SHAP values computed.")

# ══════════════════════════════════════════════════════════════════
# 1. SANITY CHECK — confirm all target MSAs are present for this year
# ══════════════════════════════════════════════════════════════════
available_msas = source_df[source_df['Year'] == EXPLAIN_YEAR]['MSA_Name'].unique()
missing = [m for m in TARGET_MSAS if m not in available_msas]
if missing:
    print(f"\nWARNING -- {len(missing)} target MSA(s) not found for {EXPLAIN_YEAR}: {missing}")
found_targets = [m for m in TARGET_MSAS if m in available_msas]
print(f"\n{len(found_targets)} of {len(TARGET_MSAS)} target MSAs found for {EXPLAIN_YEAR}.")

# ══════════════════════════════════════════════════════════════════
# 2. PER-MSA LOCAL EXPLANATION
# ══════════════════════════════════════════════════════════════════
long_records = []
wide_records = {}

for msa in found_targets:
    match = source_df[(source_df['MSA_Name'] == msa) & (source_df['Year'] == EXPLAIN_YEAR)]
    obs_idx = match.index[0]
    row_pos = source_df.index.get_loc(obs_idx)

    obs_shap = source_shap[row_pos]
    obs_features = source_X.iloc[row_pos]
    base_value = explainer.expected_value
    predicted_value = base_value + obs_shap.sum()

    print(f"\n{'-'*70}")
    print(f"{msa} — {EXPLAIN_YEAR}")
    print(f"{'-'*70}")
    print(f"  Base value: {base_value:.4f} | Predicted log(Available_SF_Total): {predicted_value:.4f}")

    local_breakdown = pd.DataFrame({
        'Feature': display_names,
        'Feature_Value': obs_features.values,
        'SHAP_Value': obs_shap,
    }).sort_values('SHAP_Value', key=np.abs, ascending=False)

    print(local_breakdown.head(10).to_string(index=False))

    for feat, val in zip(display_names, obs_shap):
        long_records.append({'MSA_Name': msa, 'Feature': feat, 'SHAP_Value': val})
    wide_records[msa] = dict(zip(display_names, obs_shap))

# ══════════════════════════════════════════════════════════════════
# 3. SAVE LONG + WIDE FORMAT CSVs
# ══════════════════════════════════════════════════════════════════
long_df = pd.DataFrame(long_records)
long_df.to_csv(OUTPUT_LONG_CSV, index=False)
print(f"\nSaved: {OUTPUT_LONG_CSV}")

wide_df = pd.DataFrame(wide_records).T  # rows = MSA, columns = feature
wide_df.to_csv(OUTPUT_WIDE_CSV)
print(f"Saved: {OUTPUT_WIDE_CSV}")

# ══════════════════════════════════════════════════════════════════
# 4. CROSS-MARKET HEATMAP — top features (by max |SHAP| across all
#    target MSAs) vs. every target MSA, side by side
# ══════════════════════════════════════════════════════════════════
top_features = wide_df.abs().max(axis=0).sort_values(ascending=False).head(15).index.tolist()
heatmap_data = wide_df[top_features]

fig, ax = plt.subplots(figsize=(14, max(6, 0.5 * len(found_targets) + 2)))
sns.heatmap(heatmap_data, cmap='RdBu_r', center=0, annot=True, fmt='.3f',
            cbar_kws={'label': 'SHAP Value'}, linewidths=0.5, ax=ax)
ax.set_xlabel("Feature")
ax.set_ylabel("MSA")
ax.set_title(
    f"Cross-Market SHAP Comparison — Top 15 Features, {EXPLAIN_YEAR}\n"
    f"Positive (red) = pushes predicted available space up | Negative (blue) = pushes it down",
    fontsize=12, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUT_HEATMAP_PNG, dpi=300, bbox_inches='tight')
plt.show()
print(f"\nSaved: {OUTPUT_HEATMAP_PNG}")

# ══════════════════════════════════════════════════════════════════
# 5. INDIVIDUAL WATERFALL PLOTS — one per MSA
# ══════════════════════════════════════════════════════════════════
for msa in found_targets:
    match = source_df[(source_df['MSA_Name'] == msa) & (source_df['Year'] == EXPLAIN_YEAR)]
    obs_idx = match.index[0]
    row_pos = source_df.index.get_loc(obs_idx)
    obs_shap = source_shap[row_pos]
    obs_features = source_X.iloc[row_pos]

    safe_name = msa.replace(',', '').replace(' ', '_').replace('-', '_')
    output_png = f"shap_local_{safe_name}_{EXPLAIN_YEAR}.png"

    plt.figure(figsize=(10, 8))
    shap.plots.waterfall(
        shap.Explanation(
            values=obs_shap,
            base_values=explainer.expected_value,
            data=obs_features.values,
            feature_names=display_names,
        ),
        max_display=15, show=False,
    )
    plt.title(f"SHAP Local Explanation — {msa}, {EXPLAIN_YEAR}", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved: {output_png}")

print(f"\n{'='*60}\nDONE — Section 7e complete\n{'='*60}")